In [ ]:
from pathlib import Path
import sys

def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start).resolve()
    for path in (start, *start.parents):
        if (path / "models").is_dir() and (path / "datasets").is_dir() and (path / "utils").is_dir():
            return path
    raise RuntimeError("Repository root was not found from the current working directory.")

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) in sys.path:
    sys.path.remove(str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT))

print("repo root:", REPO_ROOT)


In [2]:
from utils.visualization import show_result
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
import numpy as np
import csv
import os


import config_mv
import config_visa
from models.patchcore import PatchCore
from models.backbone import get_backbone
from datasets.mvtec import MyData
from datasets.visa import ViSA
from sklearn.metrics import roc_auc_score
from utils.metrics import get_image_auc
from utils.visualization import show_result, denormalize

In [3]:
if torch.cuda.is_available():
    device = 'cuda'
elif torch.backends.mps.is_available():
    device = 'mps'
else:
    device = 'cpu'
print(device)

mps


In [4]:
import os
visa_root = REPO_ROOT / "data" / "Visa"
categories = [
    folder
    for folder in os.listdir(visa_root)
    if os.path.isdir(visa_root / folder)
    and folder != "split_csv"
]
categories

['pcb3',
 'pipe_fryum',
 'pcb4',
 'pcb2',
 'candle',
 'fryum',
 'macaroni2',
 'capsules',
 'pcb1',
 'chewinggum',
 'macaroni1',
 'cashew']

In [5]:


file_path = "results.csv"
file_exists = os.path.isfile(file_path)

results = []  # ⭐ 결과 먼저 저장

for name in categories:
    train_data = ViSA(
        name,
        phase="Normal",
        batch_size=config_visa.BATCH_SIZE,
        shuffle=False,
        limit=config_visa.TRAIN_LIMIT,
    )

    test_normal = ViSA(
        name,
        phase="Normal",
        batch_size=config_visa.BATCH_SIZE,
        shuffle=False,
        limit=config_visa.TEST_LIMIT,
    )
    test_anomaly = ViSA(
        name,
        phase="Anomaly",
        batch_size=config_visa.BATCH_SIZE,
        shuffle=False,
        limit=config_visa.TEST_LIMIT,
    )
    
    backbone = get_backbone()
    patchcore = PatchCore(backbone, k=config_visa.K, device=device)

    patchcore.fit(train_data)

    scores = []
    labels = []

    for test_data in [test_normal, test_anomaly]:
        for i in range(len(test_data)):
            img, label = test_data[i]
            score, _ = patchcore.predict(img)

            scores.append(score.item())
            labels.append(label)

    # top score
    top_score = max(scores)

    # AUC
    image_auc = roc_auc_score(labels, scores)

    # ⭐ 결과 저장 (소수점 처리 포함)
    results.append([
        name,
        config_visa.TRAIN_LIMIT,
        config_visa.K,
        round(image_auc, 2),
        round(top_score, 2)
    ])

# 🔥 AUC 기준 내림차순 정렬
results = sorted(results, key=lambda x: x[3], reverse=True)

# 🔥 CSV 저장 (한 번만)
file_path = "results.csv"
with open(file_path, 'w', newline="") as f:  # 'w'로 덮어쓰기
    writer = csv.writer(f)

    writer.writerow(["category","train_limit","k","auc","top_score"])  # header 한 번만
    writer.writerows(results)

In [6]:
import pandas as pd

df = pd.read_csv("results.csv")
df

,category,train_limit,k,auc,top_score
0,pipe_fryum,30,50,1.00,7.39
1,macaroni2,30,50,1.00,7.43
2,pcb1,30,50,1.00,7.10
3,chewinggum,30,50,1.00,8.97
4,fryum,30,50,0.99,8.30
5,pcb3,30,50,0.97,7.61
6,cashew,30,50,0.96,6.53
7,pcb2,30,50,0.95,7.84
8,candle,30,50,0.95,6.39
9,macaroni1,30,50,0.84,6.01
